# ThermoFuzzy-PID · 04 · Tune and audit the adaptive controller

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fobos123deimos/thermofuzzy-pid/blob/main/notebooks/04_fuzzy_optimization.ipynb)

An explicit, reproducible hyperparameter search over fuzzy/IMC mixing and gain smoothing, with a held-out synthetic scenario and ablation.

> **Data provenance:** experiments use deterministic synthetic thermal signals and parameters reported in a 2021 undergraduate thesis. No original sensor time series have been recovered; numerical results do not reproduce or validate the published curves.

## Read the model and method visually

[Six-page vector atlas](https://fobos123deimos.github.io/thermofuzzy-pid/assets/thermal-control-atlas.pdf) · [Gallery and editable TikZ](https://github.com/fobos123deimos/thermofuzzy-pid/blob/main/docs/DIAGRAMS.md)

These original explanatory diagrams distinguish source-reported parameters, reconstruction assumptions and synthetic evidence. Cabinet geometry and sensor positions are illustrative.

![Mamdani gain scheduling: memberships, omissions and memory](https://raw.githubusercontent.com/fobos123deimos/thermofuzzy-pid/main/docs/figures/tikz/04-fuzzy-inference.png)

Three 7-by-7 sparse consequent matrices read directly from the implementation, declared input memberships, mean-of-maxima inference and separate gain memory for omitted consequents.

![Optimization: tune on training scenarios, freeze, then evaluate](https://raw.githubusercontent.com/fobos123deimos/thermofuzzy-pid/main/docs/figures/tikz/06-optimization-validation.png)

Dense-rule extensions, gain smoothing, IMC blending and a training-only differential-evolution loop with an explicit frozen boundary before held-out evaluation.


In [ ]:
# Import local editable source, or install the repository in Google Colab.
from pathlib import Path
import sys, subprocess
candidates = [Path.cwd() / "src", Path.cwd().parent / "src"]
local_src = next((p for p in candidates if (p / "burnin_pid_fuzzy").exists()), None)
if local_src is not None:
    sys.path.insert(0, str(local_src.resolve()))
else:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "git+https://github.com/fobos123deimos/thermofuzzy-pid.git"])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from burnin_pid_fuzzy.models import (Plant, Scenario, DoorEvent, PLANTS_FROM_THESIS,
                                      fopdt_step_response, simulate, simulate_methods)
from burnin_pid_fuzzy.controllers import FuzzyPID, FixedPID, imc_pid_gains
from burnin_pid_fuzzy.fuzzy import FuzzyRuleBase, infer_normalized_gains, membership_degrees
from burnin_pid_fuzzy.metrics import evaluate
np.set_printoptions(precision=4, suppress=True)


## 1. New research idea: smooth, model-assisted fuzzy scheduling

The base fuzzy scheduler implements the 2021 thesis gain mapping. An *optional research extension* interpolates fuzzy and model-based IMC gains:

$$\boldsymbol{K}^{\mathrm{hybrid}}_k=\beta\,\boldsymbol{K}^{\mathrm{fuzzy}}_k+(1-\beta)\,\boldsymbol{K}^{\mathrm{IMC}},\quad \beta\in[0,1].\tag{8}$$

Gain jumps can also be filtered:

$$\boldsymbol K'_k\leftarrow (1-\gamma)\boldsymbol K'_{k-1}+\gamma\,\boldsymbol K'_{\mathrm{MOM},k},\quad\gamma\in(0,1].\tag{9}$$

We jointly tune $(\beta,\gamma)$ under assumed synthetic conditions. **Small synthetic experiments are prone to overfitting.** This does *not* imply superior physical performance.

In [ ]:
from scipy.optimize import differential_evolution
from burnin_pid_fuzzy.models import simulate
training=[
    (Plant("train1",7.624*.90,151.6*1.10,8.78*1.20),-4.),
    (Plant("train2",7.624*1.10,151.6*.90,8.78*.80),-5.)
]
def objective(x):
    beta,gamma=map(float,x)
    penalties=[]
    for p,severity in training:
        sc=Scenario(duration=1150,dt=3,measurement_noise_std=0,
                    doors=(DoorEvent(450,60,severity),DoorEvent(840,120,severity)))
        res=simulate(FuzzyPID(mode="dense",blend_imc=beta,smoothing=gamma,dt=sc.dt),
                     p,sc,seed=2021)
        m=evaluate(res,sc,recovery_hold_s=15)
        # Units normalized so weights are interpretable.
        penalties.append(m["IAE_c_s"]/1000 + 0.01*m["actuator_total_variation"]
                         + m["overshoot_before_doors_c"]/2)
    return float(np.mean(penalties))
result=differential_evolution(objective,bounds=[(0.,1.),(.05,1.)],seed=42,
                              maxiter=2,popsize=4,polish=False)
beta,gamma=map(float,result.x)
print("Synthetic training optimum (beta,gamma):",(round(beta,3),round(gamma,3)))
print("Training objective:",round(result.fun,4),"evaluations:",result.nfev)

## 2. Held-out stress test: do not select hyperparameters on this data

We now evaluate on *different* $K$, $\tau$, $\theta$, door timing and severity, without retuning. We measure a vector of outcomes rather than collapsing the result into a single success percentage.

In [ ]:
heldout_plant=Plant("heldout",7.624*1.15,151.6*.85,8.78*1.4)
heldout=Scenario(duration=2200,dt=3,doors=(DoorEvent(640,80,-5.5),DoorEvent(1450,220,-4.5)))
choices={
    "IMC fixed":FixedPID(imc_pid_gains(),dt=3),
    "Fuzzy sparse (original rules)":FuzzyPID(mode="sparse",dt=3),
    "Fuzzy dense (smoothing .18)":FuzzyPID(mode="dense",smoothing=.18,dt=3),
    "Fuzzy hybrid (train-only tuning)":FuzzyPID(mode="dense",smoothing=gamma,blend_imc=beta,dt=3)
}
outcomes={name:simulate(ctrl,heldout_plant,heldout,seed=12) for name,ctrl in choices.items()}
report=pd.DataFrame([{"method":name,**evaluate(x,heldout)} for name,x in outcomes.items()])
print(report.round(2).to_string(index=False))
fig,ax=plt.subplots(figsize=(11,4.1))
for name,d in outcomes.items():ax.plot(d["time_s"],d["measurement_c"],label=name,lw=1.5)
ax.axhline(65,color="black",ls="--",lw=1)
for door in heldout.doors:ax.axvspan(door.start,door.start+door.duration,color="gray",alpha=.13)
ax.set(xlabel="Time (s)",ylabel="Measured temperature (°C)");ax.grid(alpha=.2)
ax.legend();plt.tight_layout();plt.show()

## 3. Extensions for a subsequent research project

- Include acceleration $\Delta^2 e[k]$ in a **3-input fuzzy system**, as suggested in Section 5 of the thesis.
- Tune missing rules with constrained Bayesian optimization or population heuristics; enforce smoothness/monotonicity and validate on held-out events.
- Compare GA/PSO/neuro-fuzzy approaches under **the same uncertainty scenarios**, computation budgets and safety constraints.
- Replace the single-zone approximation with identified multiple sensors and spatial models.
- Compare tracking quality *and* actuator energy/variation; no current claim of energy savings is validated.

All extensions must be experimentally vetted before a physical burn-in chamber is driven by the controller.